# CoastGuard AI: Coastal Erosion Susceptibility Analysis
### Extending Azzara et al. (2026, *Geomorphology*) via XGBoost & TreeSHAP

**Core Research Question:** *Can black-box machine learning (XGBoost/Random Forest) combined with SHAP achieve superior predictive accuracy over traditional MARS while offering equal or deeper interpretability for coastal protection decision-makers?*

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, classification_report
import xgboost as xgb
import shap

# Load processed dataset
df = pd.read_csv('../../dataset/processed/coastal_processed.csv')
df.head()

## 1. Exploratory Data Analysis & Feature Distributions
Analyzing the 6 key physical predictors identified in Azzara et al. (2026).

In [2]:
print(f"Total Transects: {len(df)}")
print(f"Erosion Prevalence: {df['label'].mean():.2%}")
print("\nSummary Statistics:")
df[['slope', 'storm_count', 'storm_energy', 'depth_of_closure']].describe()

## 2. Train/Test Split (70% Calibration / 30% Validation)
Adhering to the base paper's validation methodology.

In [3]:
features = ['slope', 'storm_count', 'storm_energy', 'depth_of_closure', 'geomorphology_encoded', 'longshore_encoded']
X = df[features]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
print(f"Train set: {len(X_train)} | Test set: {len(X_test)}")

## 3. Training XGBoost Classifier

In [4]:
model_xgb = xgb.XGBClassifier(
    n_estimators=180,
    max_depth=5,
    learning_rate=0.04,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42
)
model_xgb.fit(X_train, y_train)

preds_xgb = model_xgb.predict_proba(X_test)[:, 1]
auc_xgb = roc_auc_score(y_test, preds_xgb)
print(f"XGBoost Validation AUC: {auc_xgb:.4f}")

## 4. SHAP (SHapley Additive exPlanations) Interpretability
Computing game-theoretic TreeSHAP feature attributions.

In [5]:
explainer = shap.TreeExplainer(model_xgb)
shap_values = explainer.shap_values(X_test)

# Summary plot
# shap.summary_plot(shap_values, X_test, feature_names=features)
print("Top global features by mean |SHAP|:")
mean_abs_shap = np.abs(shap_values).mean(axis=0)
for f, score in sorted(zip(features, mean_abs_shap), key=lambda x: x[1], reverse=True):
    print(f" - {f:<22}: {score:.4f}")